In [108]:
#PREVIOUSLY LEARNED SETUP FOR MODEL
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model
model = init_chat_model("gpt-4o-mini", model_provider="openai")

In [109]:
#IDENTIFICATION OF ISSUE
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi! I'm Bob")])
model.invoke([HumanMessage(content="What's my name?")]).content

"I don't have access to personal data about users unless it's shared with me in the course of our conversation. So I don't know your name. How can I assist you today?"

In [110]:
#SOLUTION 1 PASS WHOLE CONVO (POOR)
#(new type of message AI message system/human/ai)
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi! I'm Bob"),
        AIMessage(content="Hello Bob! How can I assist you today?"),
        HumanMessage(content="What's my name?"),
    ]
).content

'Your name is Bob! How can I help you today, Bob?'

In [111]:
#memorysaver lets you save and restore state between graph/stores state in ram
#start marks starting point of graph
#messagestate is template for the data state of graph use
#stategraph used to build a directed graph

In [112]:
#state_schema used to define what sort of graph it is 
#state:MessagesState tells the function to expect a dictionary of messages
#state["messages"] pulls the messages list out of state

In [113]:
#.add_edge the link between 2 nodes (a -> b(its the arrow))
#.add_node adding the node/stop and running a function
#.compile makes the graph executable 

In [114]:
#MESSAGE PERSISTENCE
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

workflow = StateGraph(state_schema=MessagesState) #creation of graph and setting scheme

def call_model(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": response} #updated state also used in the next state/node in the graph (key dictionary format)

workflow.add_edge(START, "model") #tells langgraph to start the graph at the node named "model".
workflow.add_node("model", call_model) 

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)#checkpointer is a parameter expected by langraph

In [115]:
config = {"configurable": #reserved keyword in langraph
          {"thread_id": "abc123"} #custom key to track session
         }

In [116]:
#.pretty_print another way of output of convo

In [120]:
#telling name
query = "Hi! I'm Bob." 
input_messages = [HumanMessage(query)]

output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()  #last message output

#asking name
query = "What's my name?"
input_messages = [HumanMessage(query)]

output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()



================================== Ai Message ==================================

Hi again, Bob! How can I assist you today?
================================== Ai Message ==================================

Your name is Bob! What else can I help you with?


In [118]:
#messagesPlaceholder are list of messages filled in runtime

In [121]:
#PROMPT TEMPLATE
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all questions to the best of your ability in {language}."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

In [ ]:
#sequence defines what is coming string/list/dic doesnt limit content
#baseMessage means any message human/ai/system
#add_messages merges new message with previous (appends)
#Annotated lets you attach extra info/instructions to a type
#TypedDict is a way to define the expected structure

In [122]:
#BUILDING CUSTOM STATE SCHEMA 
from typing import Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict

class State(TypedDict): #state is a dic expecting this format
    messages: Annotated[Sequence[BaseMessage], add_messages] 
    language: str

In [123]:
def call_model(state: State):
    prompt = prompt_template.invoke(state)
    response = model.invoke(prompt)
    return {"messages": [response]}

workflow = StateGraph(state_schema=State) #creates a workflow/a directed graph where every state/node has these dictionary set
memory = MemorySaver()

workflow.add_edge(START, "model")
workflow.add_node("model", call_model)
app = workflow.compile(checkpointer=memory)

In [124]:
config = {"configurable": {"thread_id": "abc456"}}
query = "Hi! I'm Bob."
language = "spanish"

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages, "language": language},config)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

¡Hola Bob! ¿Cómo puedo ayudarte hoy?


In [ ]:
#trim_messages shortens conversation to fit within a token limit(manages convo)

In [137]:
#SETTING UP TRIMMER
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=70, #limit the length of messages
    strategy="last",
    token_counter=model, #these attributes are set as per requirment 
    include_system=True,
    allow_partial=False,
    start_on="human",
)

In [138]:
#sample usage of trimmer
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [139]:
workflow = StateGraph(state_schema=State)

def call_model(state: State):
    trimmed_messages = trimmer.invoke(state["messages"]) #trims messages after taking from state
    prompt = prompt_template.invoke({"messages": trimmed_messages, "language": state["language"]})
    response = model.invoke(prompt)
    return {"messages": [response]}


workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [140]:
#SHOWING THAT IT FORGOT NAME AND REMEMBERS COMPUTION AS HISTORY IS TRIMMED
config = {"configurable": {"thread_id": "abc567"}}
query = "What is my name?"
language = "English"

input_messages = messages + [HumanMessage(query)]
output = app.invoke({"messages": input_messages, "language": language},config)
output["messages"][-1].pretty_print()

config = {"configurable": {"thread_id": "abc678"}}
query = "What math problem did I ask?"
language = "English"

input_messages = messages + [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language},
    config,
)
output["messages"][-1].pretty_print()

================================== Ai Message ==================================

I don't know your name. If you'd like to share it, I’d be happy to use it!
================================== Ai Message ==================================

You asked, "What's 2 + 2?"


In [ ]:
#stream_mode tells LangGraph to stream message objects not just tokens
#tuple unpacking in the for loop

In [142]:
#STREAMING OUTPUT 
config = {"configurable": {"thread_id": "abc789"}}
query = "Hi I'm Todd, please tell me a joke."
language = "English"
input_messages = [HumanMessage(query)]

for chunk, metadata in app.stream({"messages": input_messages, "language": language},config,stream_mode="messages",):
    if isinstance(chunk, AIMessage):  #checks if streamed chunk is AI MESSAGE
        print(chunk.content, end="|")

|Hi| Todd|!| Here|’s| another| joke| for| you|:| 

|Why| don|’t| skeleton|s| fight| each| other|?| 

|They| don|’t| have| the| guts|!||